# Course Work 1

## Exercise 2

In this exercise, you will implement the Naive-Bayes algorithm for classification. Please read the instructions in the pdf provided in e-class.

In [53]:
# Here your ID: 201932
import numpy as np
import matplotlib.pyplot as plt #can be reeplaced by other libraries
import pandas as pd
#is not allowed to use libraries besides the ones here besides reeplacing matplotlib

$
P(C_k | \mathbf{x}) = \frac{p(\mathbf{x} | C_k) \cdot P(C_k)}{p(\mathbf{x})}
$

#### Idea: Make the simplifying assumption that attribute values ​​are conditionally independent given target values

$
p(\mathbf{x} | C_i) = p(x_1, x_2, \ldots, x_n | C_i)
$

$
\approx p(x_1 | C_i) \cdot p(x_2 | C_i) \cdot \ldots \cdot p(x_n | C_i)
$

$
= \prod_{j} p(x_j | C_i)
$

In [54]:
def standar_norm(X):
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)
    X_norm = (X - X_mean) / X_std
    return X_norm

def verify_columns(df):
    feature_columns = df.columns[:-1]
    return feature_columns

def split_data(df, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    train_indices = []
    test_indices = []

    # Indicate last column as label
    for label in df[df.columns[-1]].unique():
        label_indices = df[df[df.columns[-1]] == label].index.to_numpy()
        np.random.shuffle(label_indices)

        n_test = int(len(label_indices) * test_size)
        test_indices.extend(label_indices[:n_test])
        train_indices.extend(label_indices[n_test:])

    df_train = df.loc[train_indices].reset_index(drop=True)
    df_test = df.loc[test_indices].reset_index(drop=True)
    return df_train, df_test

def read_csv(filename):
    df = pd.read_csv(filename)
    return df


In [55]:
def calculate_priors(y):
    classes, counts = np.unique(y, return_counts=True)
    priors = counts / counts.sum()
    return classes, priors

def calculate_means_and_variances(X, y, classes):
    #μ_cj = (1/N_c) * Σ_{i:y_i=c} x_{ij} (mean) 
    #σ_cj^2 = (1/(N_c-1)) * Σ_{i:y_i=c} (x_{ij} - μ_cj)^2 (sample variance)
    n_classes = len(classes)
    n_features = X.shape[1]
    means = np.zeros((n_classes, n_features))
    variances = np.zeros((n_classes, n_features))

    for i, c in enumerate(classes):
        X_c = X[y == c]
        means[i, :] = X_c.mean(axis=0)
        variances[i, :] = X_c.var(axis=0, ddof=1)

    return means, variances

def gaussian_pdf(x, mean, var):
    # Gaussian probability density function
    # f(x_j | μ_j, σ_j^2) = (1 / sqrt(2πσ_j^2)) * exp( - (x_j - μ_j)^2 / (2σ_j^2) )
    var = np.maximum(var, 1e-9) # this is to avoid division by zero
    numerator = np.exp(-((x - mean) ** 2) / (2 * var))
    denominator = np.sqrt(2 * np.pi * var)
    return numerator / denominator

def naive_bayes(X_train, y_train, X_test):
    classes, priors = calculate_priors(y_train)
    means, variances = calculate_means_and_variances(X_train, y_train, classes)
    predictions_test, _ = predict(X_test, classes, priors, means, variances)
    return predictions_test

def predict(X, classes, priors, means, variances):
    # log P(C=c|x) ∝ log P(C=c) + Σ_{j=1}^d log f(x_j | μ_cj, σ_cj^2
    predictions = []
    confidences = []
    
    for x in X:
        posteriors = []
        for i, c in enumerate(classes):
            prior = np.log(priors[i]) # log P(C=c)
            likelihood = np.sum(np.log(gaussian_pdf(x, means[i], variances[i])))
            posterior = prior + likelihood
            posteriors.append(posterior)
        
        posteriors = np.array(posteriors)
        # Here I want to avoid numerical overflow/underflow issues
        posteriors = posteriors - np.max(posteriors)
        exp_posteriors = np.exp(posteriors)
        probabilities = exp_posteriors / np.sum(exp_posteriors)
        
        pred_idx = np.argmax(probabilities)
        predictions.append(classes[pred_idx])
        confidences.append(probabilities[pred_idx])
    
    return np.array(predictions), np.array(confidences)


def calculate_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def calculate_precision(y_true, y_pred, classes):
    precision = {}
    for c in classes:
        true_positives = np.sum((y_true == c) & (y_pred == c))
        predicted_positives = np.sum(y_pred == c)
        precision[c] = true_positives / predicted_positives if predicted_positives > 0 else 0
    return precision

def calculate_recall(y_true, y_pred, classes):
    recall = {}
    for c in classes:
        true_positives = np.sum((y_true == c) & (y_pred == c))
        actual_positives = np.sum(y_true == c)
        recall[c] = true_positives / actual_positives if actual_positives > 0 else 0
    return recall

def print_metrics(y_true, y_pred, dataset_name):
    classes = np.unique(y_true)
    accuracy = calculate_accuracy(y_true, y_pred)
    precision = calculate_precision(y_true, y_pred, classes)
    recall = calculate_recall(y_true, y_pred, classes)
    
    print(f"\n=== EVALUACIÓN EN {dataset_name.upper()} ===")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"\nPrecision por clase:")
    for c in classes:
        print(f"  Clase {c}: {precision[c]:.4f}")
    print(f"\nRecall por clase:")
    for c in classes:
        print(f"  Clase {c}: {recall[c]:.4f}")

In [56]:
df = read_csv('diabetes.csv')
# Normalize the features minus the label column
feature_columns = verify_columns(df)
df[feature_columns] = standar_norm(df[feature_columns])

df_train, df_test = split_data(df, test_size=0.2, random_state=42)
df_train, df_val = split_data(df_train, test_size=0.5, random_state=42)


print("Training set shape:", df_train.shape)
print("Validation set shape:", df_val.shape)
print("Test set shape:", df_test.shape)
print(df_train)

X_train, y_train = df_train[feature_columns].values, df_train[df.columns[-1]].values
X_val, y_val = df_val[feature_columns].values, df_val[df.columns[-1]].values
X_test, y_test = df_test[feature_columns].values, df_test[df.columns[-1]].values

predictions_val = naive_bayes(X_train, y_train, X_val)
print_metrics(y_val, predictions_val, "VALIDATION")

X_train_combined = np.vstack([X_train, X_val])
y_train_combined = np.hstack([y_train, y_val])

print(f"Combined training set shape: {X_train_combined.shape}")
print(f"Test set shape: {X_test.shape}")

predictions_test = naive_bayes(X_train_combined, y_train_combined, X_test)
print_metrics(y_test, predictions_test, "TEST (FINAL EVALUATION)")


Training set shape: (308, 9)
Validation set shape: (307, 9)
Test set shape: (153, 9)
     Pregnancies   Glucose  BloodPressure  SkinThickness   Insulin       BMI  \
0       0.342981  0.284975       0.666618      -1.288212 -0.692891  0.902069   
1       1.233880  2.350587       0.356432       0.530902  1.738320  0.698998   
2      -1.141852 -0.434859      -0.367337       0.593630 -0.050356  0.584771   
3      -0.844885  1.317781       0.149641      -1.288212 -0.692891  0.889377   
4       0.046014  0.660541       0.873409      -1.288212 -0.692891  1.523973   
..           ...       ...            ...            ...       ...       ...   
303     1.233880 -0.653939       0.356432      -1.288212 -0.692891  0.851301   
304    -0.250952  1.849832      -0.263941       0.279989 -0.085088  0.254780   
305    -0.547919 -1.217288      -0.884314       0.091805  0.305642 -0.443275   
306    -0.250952 -0.184482      -0.160546       1.158182  0.522715  0.775149   
307    -0.547919 -0.966911      -0.